#Setup (run once before program)

In [ ]:
#installing faiss
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.4 MB/s eta 0:00:00


In [ ]:
#sentence transformer for the model
!pip install -U sentence-transformers

In [ ]:
#getting a small portion of the dataset
from datasets import load_dataset
dataset = load_dataset("microsoft/ms_marco", "v2.1", split="train[:10000]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

v2.1/validation-00000-of-00001.parquet:   0%|          | 0.00/210M [00:00<?, ?B/s]

v2.1/train-00000-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

v2.1/train-00001-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

...

v2.1/train-00006-of-00007.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

v2.1/test-00000-of-00001.parquet:   0%|          | 0.00/204M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/101093 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/808731 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/101092 [00:00<?, ? examples/s]

#Main Code

##Libraries

In [ ]:
import numpy as np
import pandas as pd
import time

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss

##Dataset

In [ ]:
#taking a look at the data
print(dataset, end='\n\n')
dataset[0]

Dataset({
    features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
    num_rows: 10000
})



{'answers': ['The immediate impact of the success of the manhattan project was the only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.'],
 'passages': {'is_selected': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  'passage_text': ['The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.',
   'The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science.',
   'Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of th

In [ ]:
#cleaning up the data and removing layers so that it is just a simple list for each row
documents = []
for row in dataset:

  passages = row["passages"]["passage_text"]
  is_selected = row["passages"]["is_selected"]

  for text, label in zip(passages, is_selected):
    documents.append({
      "query_id": row["query_id"],
      "query": row["query"],
      "text": text,
      "relevant": label
    })

df = pd.DataFrame(documents)

print("Passages:", len(df))
df.head()


Passages: 99736


,query_id,query,text,relevant
0,1185869,)what was the immediate impact of the success ...,The presence of communication amid scientific ...,1
1,1185869,)what was the immediate impact of the success ...,The Manhattan Project and its atomic bomb help...,0
2,1185869,)what was the immediate impact of the success ...,Essay on The Manhattan Project - The Manhattan...,0
3,1185869,)what was the immediate impact of the success ...,The Manhattan Project was the name for a proje...,0
4,1185869,)what was the immediate impact of the success ...,versions of each volume as well as complementa...,0


In [ ]:
#taking away all the empty passages if there are any
df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("Passages after cleaning:", len(df))

Passages after cleaning: 99736


##Baseline Model

###Setting up the Model

In [ ]:
#using all-MiniLM-L6-v2 as suggested in the task doc
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

###Embedding the Texts

In [ ]:
#turning the texts into vectors
passage_texts = df["text"].tolist()

start = time.time()
passage_embeddings = model.encode(passage_texts,show_progress_bar=True)
end = time.time()

print("Embedding shape:", passage_embeddings.shape)
print("Time taken:", end - start)
#I went back and added a time count becuase it took so long and I wanted to see how long it took

Batches:   0%|          | 0/3117 [00:00<?, ?it/s]

Embedding shape: (99736, 384)
Time taken: 4463.885945081711


In [ ]:
def semantic_search(query, top=10):
  query_embedding = model.encode([query]).astype("float32")
  faiss.normalize_L2(query_embedding)

  scores, indices = index.search(query_embedding, top)

  results = []
  for id, score in zip(indices[0], scores[0]):
    results.append({
      "query": query,
      "score": float(score),
      "text": df.iloc[id]["text"][:200],
      "relevant_label": int(df.iloc[id]["relevant"])
    })

  return results

##Tuned Model

In [ ]:
from sentence_transformers import InputExample
import random

train_examples = []

#taking only the first 500 queries cos lite
unique_query_ids = df["query_id"].unique()[:500]

for i in unique_query_ids:
  selected = df[df["query_id"] == i]

  positives = selected[selected["relevant"] == 1]
  negatives = selected[selected["relevant"] == 0]

  if len(positives) == 0 or len(negatives) == 0:
    continue

  pos_row = positives.sample(1).iloc[0]
  neg_row = negatives.sample(1).iloc[0]

  #positive match yayy
  train_examples.append(InputExample(texts=[pos_row["query"], pos_row["text"]]))

  #negative pair
  train_examples.append(InputExample(texts=[pos_row["query"], neg_row["text"]]))

print("Total training pairs:", len(train_examples))


Total training pairs: 610


In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import losses

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

train_loss = losses.MultipleNegativesRankingLoss(model)

#doing only one epoch to save time
model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=1, warmup_steps=100, show_progress_bar=True)


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


In [ ]:
#encoding the passages again using the tuned model
passage_embeddings_ft = model.encode(df["text"].tolist(), show_progress_bar=True)

passage_embeddings_ft = np.array(passage_embeddings_ft).astype("float32")
faiss.normalize_L2(passage_embeddings_ft)

index_ft = faiss.IndexFlatIP(passage_embeddings_ft.shape[1])
index_ft.add(passage_embeddings_ft)

print("Fine-tuned FAISS index ready")

Batches:   0%|          | 0/3117 [00:00<?, ?it/s]

Fine-tuned FAISS index ready
